# One Schema Reload

Truncates all tables in a schema and reloads data from CSV files.
Run this notebook to do a clean reload for a single schema.


In [ ]:
# ── Configuration ──────────────────────────────────────────────
# Edit SCHEMA_NAME to target a different schema
SCHEMA_NAME = "customer"
# ───────────────────────────────────────────────────────────────

print(f"Target schema: {SCHEMA_NAME}")


In [ ]:
import os

# Truncate all tables in the schema
root = "/lakehouse/default/Tables"
EXCLUDE_EXTENSIONS = (".gz", ".delta")
EXCLUDE_ENTRIES = {"_delta_log", "_committed", "_started"}

schema_path = os.path.join(root, SCHEMA_NAME)

if not os.path.isdir(schema_path):
    raise Exception(f"Schema '{SCHEMA_NAME}' not found at {schema_path}")

tables = sorted([
    t for t in os.listdir(schema_path)
    if t not in EXCLUDE_ENTRIES and not any(t.endswith(ext) for ext in EXCLUDE_EXTENSIONS)
])

print(f"Truncating {len(tables)} tables in schema '{SCHEMA_NAME}'...")
for table in tables:
    spark.sql(f"TRUNCATE TABLE {SCHEMA_NAME}.{table}")
    print(f"  ✅ Truncated: {SCHEMA_NAME}.{table}")

print(f"\nAll tables truncated.")


In [ ]:
# Load data - calls the schema-specific load notebook
LOAD_NOTEBOOKS = {
    "customer": "data_processing/load_customer",
}

load_notebook = LOAD_NOTEBOOKS.get(SCHEMA_NAME)
if not load_notebook:
    raise Exception(f"No load notebook configured for schema '{SCHEMA_NAME}'")

print(f"Running load notebook: {load_notebook}")
mssparkutils.notebook.run(load_notebook)
print(f"\n🎉 Schema '{SCHEMA_NAME}' reload complete!")
